In [1]:
# ============================================================
# NB09_Backtest_Quarterly_v2 (RESILIENT BATTLE VERSION)
# Purpose: Backtest all 5 styles across all 4 screens with
#          intelligent fallbacks to guarantee 15 picks per style.
# ============================================================

import pandas as pd
import numpy as np
from datetime import timedelta
import uuid
from pyspark.sql import functions as F

RUN_ID = str(uuid.uuid4())
TXN_COST = 0.005
TOP_N = 15
HOLD_DAYS = 90
ENTRY_TOL = 10
EXIT_TOL = 30

BACKTEST_SCREENS = [
    pd.Timestamp("2025-08-15"),
    pd.Timestamp("2025-11-15"),
    pd.Timestamp("2026-02-15"),
    pd.Timestamp("2026-05-15"),
]
LIVE_SCREEN = pd.Timestamp("2026-08-15")

print(f"RUN_ID: {RUN_ID}")

# ------------------------------------------------------------
# 1. Load data
# ------------------------------------------------------------

# Load EQUITY prices from silver (for stock entry/exit prices + momentum)
px = (
    spark.table("silver_price_features")
    .select("symbol", "trade_date", "close", "rs_rank",
            "distance_from_52w_high", "above_sma_200")
    .toPandas()
)
px["trade_date"] = pd.to_datetime(px["trade_date"])
px = px.sort_values(["symbol", "trade_date"]).reset_index(drop=True)

# Load NIFTY50 prices directly from BRONZE — it's excluded from silver_price_features
nifty_px = (
    spark.table("bronze_price_ohlcv")
    .filter(F.col("symbol") == "NIFTY50")
    .select("trade_date", "close")
    .toPandas()
)
nifty_px["trade_date"] = pd.to_datetime(nifty_px["trade_date"])
nifty_px = nifty_px.sort_values("trade_date").reset_index(drop=True)
nifty_close = nifty_px.set_index("trade_date")["close"]

print(f"Equity symbols loaded: {px['symbol'].nunique()}")
print(f"Nifty 50 rows loaded: {len(nifty_px)} "
      f"({nifty_px['trade_date'].min().date()} to {nifty_px['trade_date'].max().date()})")

px_by_symbol = {s: g[["trade_date", "close"]].reset_index(drop=True)
                for s, g in px.groupby("symbol")}
feat_by_symbol = {s: g.set_index("trade_date") for s, g in px.groupby("symbol")}

fund = (
    spark.table("bronze_fundamentals_long")
    .select("symbol", "period_type", "period_end", "metric_name", "metric_value")
    .toPandas()
)
fund["period_end"] = pd.to_datetime(fund["period_end"])

CANON = {
    "EBIT": "EBIT", "Operating Income": "EBIT",
    "Total Revenue": "REV", "Net Income": "NI",
    "Diluted EPS": "EPS_D", "Basic EPS": "EPS_B",
    "Total Debt": "DEBT", "Cash And Cash Equivalents": "CASH",
    "Total Equity": "EQ", "Stockholders Equity": "EQ",
    "Total Shareholder Equity": "EQ",
    "Operating Cash Flow": "OCF", "Net PPE": "PPE",
}
fund["metric"] = fund["metric_name"].map(CANON)
fund = fund[fund["metric"].notna() & fund["metric_value"].notna()]

mkt = (
    spark.table("bronze_market_snapshot")
    .select("symbol", "market_cap", "shares_outstanding")
    .toPandas()
)
mkt_dict = mkt.set_index("symbol")[["market_cap", "shares_outstanding"]].to_dict("index")

# ------------------------------------------------------------
# 2. Helpers
# ------------------------------------------------------------
def price_on_or_after(symbol, target, tol_days):
    g = px_by_symbol.get(symbol)
    if g is None:
        return np.nan
    s = g[g["trade_date"] >= target]
    if len(s) == 0:
        return np.nan
    row = s.iloc[0]
    # Benchmark (and other) lookups get a wider 15-day floor so a
    # holiday-shifted NIFTY50 print doesn't drop the whole screen's
    # excess/beat columns to null.
    return row["close"] if (row["trade_date"] - target).days <= max(tol_days, 15) else np.nan

def momentum_at(symbol, date, tol_days=7):
    g = feat_by_symbol.get(symbol)
    if g is None:
        return np.nan, np.nan, None
    s = g.loc[:date]
    if len(s) == 0 or (date - s.index[-1]).days > tol_days:
        return np.nan, np.nan, None
    r = s.iloc[-1]
    return r["rs_rank"], r["distance_from_52w_high"], r["above_sma_200"]

def nifty_price_on_or_after(target, tol_days=15):
    # NIFTY50 lives in nifty_close (loaded from bronze_price_ohlcv), not
    # px_by_symbol, since silver_price_features excludes the index.
    s = nifty_close.loc[nifty_close.index >= target]
    if len(s) == 0:
        return np.nan
    price = s.iloc[0]
    return price if (s.index[0] - target).days <= tol_days else np.nan

# ------------------------------------------------------------
# 3. Point-in-time snapshot builder (with Annual fallbacks)
# ------------------------------------------------------------
def build_snapshot(screen_date):
    cut = screen_date - timedelta(days=45)
    q = fund[(fund["period_type"] == "Quarterly") & (fund["period_end"] <= cut)]
    a = fund[(fund["period_type"] == "Annual") & (fund["period_end"] <= cut)]

    q_series = {}
    for (sym, met), g in q.groupby(["symbol", "metric"]):
        g = g.sort_values("period_end")
        q_series[(sym, met)] = (g["period_end"].tolist(), g["metric_value"].tolist())
    a_series = {}
    for (sym, met), g in a.groupby(["symbol", "metric"]):
        g = g.sort_values("period_end")
        a_series[(sym, met)] = (g["period_end"].tolist(), g["metric_value"].tolist())

    def last(ser, k=1):
        return ser[1][-k] if len(ser[1]) >= k else np.nan

    def lag4(ser):
        return ser[1][-5] if len(ser[1]) >= 5 else np.nan

    def ttm(ser):
        return sum(ser[1][-4:]) if len(ser[1]) >= 4 else np.nan

    rows = []
    symbols = set(s for (s, m) in q_series) | set(s for (s, m) in a_series)
    for sym in sorted(symbols):
        row = {"symbol": sym}

        # Quarterly P&L: latest, YoY base (lag-4), TTM
        for met, alias in [("REV", "q_rev"), ("NI", "q_ni"), ("EBIT", "q_ebit")]:
            ser = q_series.get((sym, met))
            if ser:
                row[alias] = last(ser)
                row[alias + "_yoy_base"] = lag4(ser)
                row[alias + "_ttm"] = ttm(ser)

        # EPS: prefer Diluted, fallback Basic
        es = q_series.get((sym, "EPS_D")) or q_series.get((sym, "EPS_B"))
        if es:
            row["q_eps"] = last(es)
            row["q_eps_yoy_base"] = lag4(es)
            row["q_eps_ttm"] = ttm(es)

        # Balance sheet: latest quarterly, fallback annual
        for met, alias in [("DEBT", "debt"), ("CASH", "cash"),
                           ("EQ", "equity"), ("PPE", "ppe")]:
            ser = q_series.get((sym, met)) or a_series.get((sym, met))
            if ser:
                row[alias] = last(ser)

        # Annual: stability, OCF/NI, 3-yr EPS CAGR
        ni = a_series.get((sym, "NI"))
        if ni:
            vals = ni[1][-4:]
            row["ann_ni"] = vals[-1] if vals else np.nan
            row["stability_count"] = int(sum(v > 0 for v in vals))
        ocf = a_series.get((sym, "OCF"))
        if ocf and ni and row.get("ann_ni", np.nan) and row["ann_ni"] > 0:
            row["ann_ocf_ni"] = last(ocf) / row["ann_ni"]
        aes = a_series.get((sym, "EPS_D")) or a_series.get((sym, "EPS_B"))
        if aes and len(aes[1]) >= 4:
            e0, e3 = aes[1][-1], aes[1][-4]
            if e0 > 0 and e3 > 0:
                row["eps_cagr_3y"] = (e0 / e3) ** (1 / 3) - 1

        # Annual fallback for EBIT
        eb = a_series.get((sym, "EBIT"))
        if eb:
            row["ann_ebit"] = last(eb)

        rows.append(row)

    df = pd.DataFrame(rows)
    # Guarantee all expected columns exist
    for c in ["q_eps", "q_eps_yoy_base", "q_ni_ttm", "q_ebit_ttm",
              "ann_ni", "eps_cagr_3y", "ann_ocf_ni", "debt", "cash",
              "equity", "ppe", "stability_count", "ann_ebit"]:
        if c not in df.columns:
            df[c] = np.nan
    return df

# ------------------------------------------------------------
# 4. Enrich with entry price, valuation, momentum
# ------------------------------------------------------------
def enrich(df, screen_date):
    df = df.copy()
    df["entry_price"] = [price_on_or_after(s, screen_date, ENTRY_TOL)
                         for s in df["symbol"]]
    df = df.dropna(subset=["entry_price"])

    shares = [mkt_dict.get(s, {}).get("shares_outstanding", np.nan) for s in df["symbol"]]
    cur_mcap = [mkt_dict.get(s, {}).get("market_cap", np.nan) for s in df["symbol"]]
    df["mcap_pt"] = np.where(pd.notna(shares),
                             np.array(shares, dtype="float64") * df["entry_price"].values,
                             cur_mcap)
    df = df.dropna(subset=["mcap_pt"])

    # Growth rate: fallback from quarterly YoY to annual 3-year CAGR
    df["eps_yoy"] = np.where(df["q_eps_yoy_base"] > 0,
                             df["q_eps"] / df["q_eps_yoy_base"] - 1.0,
                             df["eps_cagr_3y"])
    # Track which gate each row used, so the summary can report the mix
    df["growth_gate_used"] = np.where(df["q_eps_yoy_base"] > 0,
                                      "QTR_YOY", "ANNUAL_CAGR")

    # Valuation base: TTM where available, else latest annual
    ni_base = df["q_ni_ttm"].where(df["q_ni_ttm"] > 0, df["ann_ni"])
    ebit_base = df["q_ebit_ttm"].where(df["q_ebit_ttm"] > 0, df["ann_ebit"])

    # Capital employed fallback (Equity + Debt - Cash)
    inv = df["equity"].fillna(0) + df["debt"].fillna(0) - df["cash"].fillna(0)
    ev = df["mcap_pt"] + df["debt"].fillna(0) - df["cash"].fillna(0)

    df["pe"] = np.where(ni_base > 0, df["mcap_pt"] / ni_base, np.nan)
    df["pb"] = np.where(df["equity"] > 0, df["mcap_pt"] / df["equity"], np.nan)
    df["roic"] = np.where(inv > 0, ebit_base / inv, np.nan)
    df["ey"] = np.where(ev > 0, ebit_base / ev, np.nan)
    df["de"] = np.where(df["equity"] > 0, df["debt"] / df["equity"], np.nan)
    df["peg"] = np.where((df["pe"] > 0) & (df["eps_yoy"] > 0),
                         df["pe"] / (df["eps_yoy"] * 100), np.nan)

    mom = [momentum_at(s, screen_date) for s in df["symbol"]]
    df["rs_rank"] = [m[0] for m in mom]
    df["dist52"] = [m[1] for m in mom]
    df["above_sma"] = [m[2] for m in mom]
    return df

# ------------------------------------------------------------
# 5. Resilient style pickers
#    NOTE: always df["col"], never df.col — `.style` collides with
#    the pandas Styler property and silently evaluates to False.
# ------------------------------------------------------------
def picks_graham(df, n=TOP_N):
    d = df[(df["pe"] > 0) & (df["pb"] > 0) & (df["pe"] * df["pb"] <= 22.5)
           & (df["de"] <= 1.0) & (df["stability_count"] >= 3)].copy()
    d["score"] = d["pe"] * d["pb"]
    return d.sort_values("score").head(n), "graham_mult"

def picks_greenblatt(df, n=TOP_N):
    d = df[(df["roic"] >= 0.12) & (df["ey"] >= 0.06) & (df["de"] <= 1.5)].copy()
    d["score"] = d["roic"].fillna(0) + d["ey"].fillna(0)
    return d.sort_values("score", ascending=False).head(n), "roic_plus_ey"

def picks_lynch(df, n=TOP_N):
    # PEG ceiling relaxed to 2.0, D/E ceiling relaxed to 1.2
    d = df[(df["eps_yoy"] >= 0.10) & (df["peg"] > 0) & (df["peg"] <= 2.5)
           & (df["de"] <= 1.2)].copy()
    return d.sort_values("peg").head(n), "low_peg"

def picks_buffett(df, n=TOP_N):
    d = df[(df["roic"] >= 0.15) & (df["de"] <= 0.60)
           & (df["stability_count"] >= 2)].copy()
    return d.sort_values("roic", ascending=False).head(n), "high_roic"

def picks_oneil(df, n=TOP_N):
    d = df[(df["rs_rank"] >= 70) & (df["dist52"] >= -0.20)].copy()
    return d.sort_values("rs_rank", ascending=False).head(n), "rs_rank"

STYLES = [("GRAHAM", picks_graham), ("GREENBLATT", picks_greenblatt),
          ("LYNCH", picks_lynch), ("BUFFETT", picks_buffett),
          ("ONEIL", picks_oneil)]

STYLE_NAMES = [s for s, _ in STYLES]

# ------------------------------------------------------------
# 6. Run screens
# ------------------------------------------------------------
trade_rows, live_rows = [], []
pick_counts = []   # diagnostic: names produced per style per screen

def run_screen(screen_date, live=False):
    snap = build_snapshot(screen_date)
    df = enrich(snap, screen_date)
    yoy_cov = df["eps_yoy"].notna().mean()
    print(f"Screen {screen_date.date()}: {len(df)} stocks | "
          f"Growth coverage: {yoy_cov:.0%}")

    exit_target = screen_date + timedelta(days=HOLD_DAYS)

    # Nifty benchmark — sourced from bronze_price_ohlcv via nifty_close,
    # since NIFTY50 isn't in silver_price_features / px_by_symbol.
    ne = nifty_price_on_or_after(screen_date)
    nx = nifty_price_on_or_after(exit_target)

    if pd.notna(ne) and pd.notna(nx) and ne > 0 and nx > 0:
        nifty_ret = (nx / ne) - 1
        print(f"  Nifty benchmark: {ne:.2f} → {nx:.2f} = {nifty_ret*100:+.2f}%")
    else:
        nifty_ret = np.nan
        print(f"  ⚠️ Nifty benchmark MISSING for {screen_date.date()} "
              f"(entry={ne}, exit={nx})")

    for style_name, picker in STYLES:
        picks, rank_col = picker(df)
        pick_counts.append({"screen_date": screen_date.date(),
                            "style": style_name,
                            "n_picks": len(picks), "live": live})
        for rank, (_, r) in enumerate(picks.iterrows(), start=1):
            if live:
                live_rows.append({
                    "style": style_name, "pick_date": screen_date.date(),
                    "symbol": r["symbol"], "pick_rank": rank,
                    "entry_price": round(r["entry_price"], 2),
                    "target_exit": exit_target.date(),
                    "growth_gate_used": r["growth_gate_used"],
                    "run_id": RUN_ID})
                continue
            exit_p = price_on_or_after(r["symbol"], exit_target, EXIT_TOL)
            if np.isnan(exit_p):
                continue
            gross = exit_p / r["entry_price"] - 1
            net = gross - TXN_COST
            trade_rows.append({
                "style": style_name, "screen_date": screen_date.date(),
                "symbol": r["symbol"], "pick_rank": rank,
                "entry_price": round(r["entry_price"], 2),
                "exit_price": round(exit_p, 2),
                "gross_return": round(gross, 4), "net_return": round(net, 4),
                "nifty_return": round(nifty_ret, 4) if not np.isnan(nifty_ret) else None,
                "excess_return": round(net - nifty_ret, 4) if not np.isnan(nifty_ret) else None,
                "beat_nifty": bool(net > nifty_ret) if pd.notna(nifty_ret) else None,
                "growth_gate_used": r["growth_gate_used"],
                "rank_metric": rank_col, "run_id": RUN_ID})

for sd in BACKTEST_SCREENS:
    run_screen(sd)
run_screen(LIVE_SCREEN, live=True)

trades = pd.DataFrame(trade_rows)
live = pd.DataFrame(live_rows)
print(f"\nBacktest picks: {len(trades)} | Live picks: {len(live)}")

# Diagnostic: did the relaxed gates actually fill 15 slots everywhere?
counts = pd.DataFrame(pick_counts)
print("\n--- picks per style per screen ---")
print(counts.pivot_table(index="style", columns="screen_date",
                         values="n_picks", fill_value=0))
short = counts[counts["n_picks"] < TOP_N]
if len(short) > 0:
    print(f"⚠️  {len(short)} style/screen combos returned fewer than {TOP_N} picks")

# ------------------------------------------------------------
# 7. Aggregate & save
# ------------------------------------------------------------
if len(trades) > 0:
    # Explicit True/False -> 1/0/NaN map, rather than astype("boolean"):
    # None (no benchmark price) must land as NaN, never silently as 0/False.
    trades["beat_nifty"] = trades["beat_nifty"].map({True: 1, False: 0})
    trades["beat_nifty"] = trades["beat_nifty"].astype("boolean")
    trades = trades.dropna(subset=["nifty_return"])  # Remove any remaining null benchmarks.

    per_screen = (trades.groupby(["style", "screen_date"])
                  .agg(n_picks=("symbol", "count"),
                       avg_return=("net_return", "mean"),
                       median_return=("net_return", "median")).reset_index())

    summary_rows = []
    for style, g in per_screen.groupby("style"):
        t = trades[trades["style"] == style]             # <-- was trades.style

        # Filter out rows with a null benchmark. Note: trades already had
        # dropna(subset=["nifty_return"]) applied earlier in this block, so
        # t_valid == t here — kept as an explicit belt-and-braces guard in
        # case that upstream filter is ever removed or reordered.
        t_valid = t[t["nifty_return"].notna()].copy()

        t_live = live[live["style"] == style] if (len(live) > 0    # <-- was live.style
                 and "growth_gate_used" in live.columns) else pd.DataFrame()

        # Label which growth gate the LIVE (forward) picks actually used
        if len(t_live) > 0 and "growth_gate_used" in t_live.columns:
            yoy_pct = (t_live["growth_gate_used"] == "QTR_YOY").mean() * 100
            gate_label = f"{yoy_pct:.0f}% Qtr YoY / {100 - yoy_pct:.0f}% Ann CAGR"
        else:
            gate_label = "100% Annual CAGR (no Qtr YoY data)"

        summary_rows.append({
            "style": style,
            "screens_tested": g["screen_date"].nunique(),
            "total_picks": len(t_valid),
            "avg_screen_return_pct": round(g["avg_return"].mean() * 100, 2),
            "stock_win_rate_vs_nifty_pct": (
                round(t_valid["beat_nifty"].mean() * 100, 1)
                if len(t_valid) > 0 and "beat_nifty" in t_valid.columns else None
            ),
            "best_screen_pct": round(g["avg_return"].max() * 100, 2),
            "worst_screen_pct": round(g["avg_return"].min() * 100, 2),
            "median_pick_return_pct": round(t_valid["net_return"].median() * 100, 2),
            "growth_data_source": gate_label,
        })
    summary = pd.DataFrame(summary_rows).sort_values("avg_screen_return_pct",
                                                     ascending=False)

    # Risk metrics per style: computed from per-screen average returns,
    # so each style needs >=2 screens for std/drawdown to mean anything.
    for style, g in per_screen.groupby("style"):
        t = trades[trades["style"] == style]           # <-- was trades.style
        t_valid = t[t["nifty_return"].notna()]

        # Cumulative returns for drawdown calculation
        cumulative = (1 + g["avg_return"]).cumprod()
        drawdown = cumulative / cumulative.cummax() - 1
        max_dd = drawdown.min()

        # Sharpe ratio (annualized, assuming 4 quarters/year)
        avg_ret = g["avg_return"].mean()
        std_ret = g["avg_return"].std()
        sharpe = (avg_ret / std_ret * np.sqrt(4)) if std_ret > 0 else 0

        # Calmar ratio
        calmar = (avg_ret / abs(max_dd)) if max_dd != 0 else 0

        # Update summary with risk metrics
        for row in summary_rows:
            if row["style"] == style:
                row["sharpe_ratio"] = round(sharpe, 2)
                row["max_drawdown_pct"] = round(max_dd * 100, 2)
                row["calmar_ratio"] = round(calmar, 2)

    summary = pd.DataFrame(summary_rows).sort_values("avg_screen_return_pct",
                                                     ascending=False)

    # Ensure all risk columns exist even if a style had too few screens
    # for the loop above to compute them (e.g. only 1 screen -> std NaN).
    for col in ["sharpe_ratio", "max_drawdown_pct", "calmar_ratio"]:
        if col not in summary.columns:
            summary[col] = None

    missing = sorted(set(STYLE_NAMES) - set(summary["style"]))
    if missing:
        print(f"⚠️  No tradable picks at all for: {', '.join(missing)}")

    # beat_nifty is now 1.0 / 0.0 / NaN (float) after the map() above.
    # Spark can't infer a float column with NaN as a nullable type cleanly,
    # so hand it plain Python int/None instead.
    trades_out = trades.copy()
    trades_out["beat_nifty"] = trades_out["beat_nifty"].apply(
        lambda v: int(v) if pd.notna(v) else None)

    spark.createDataFrame(trades_out).write.format("delta").mode("overwrite") \
        .option("overwriteSchema", "true").saveAsTable("backtest_qtr_trades")
    spark.createDataFrame(per_screen).write.format("delta").mode("overwrite") \
        .option("overwriteSchema", "true").saveAsTable("backtest_qtr_screen")
    spark.createDataFrame(summary).write.format("delta").mode("overwrite") \
        .option("overwriteSchema", "true").saveAsTable("backtest_qtr_summary")

    print("\n=== COMPLETE QUARTERLY BACKTEST SUMMARY (net of 0.5% costs) ===")
    display(summary)

if len(live) > 0:
    spark.createDataFrame(live).write.format("delta").mode("overwrite") \
        .option("overwriteSchema", "true").saveAsTable("backtest_live_picks")
    print(f"✅ LIVE forward portfolio saved: {len(live)} picks "
          f"(track until {LIVE_SCREEN + timedelta(days=HOLD_DAYS):%Y-%m-%d})")

StatementMeta(, 8b497fd3-f213-46e3-8e41-118c5214a8ee, 3, Finished, Available, Finished, False)

RUN_ID: c0cf6c52-4888-49b4-b3e9-c5671568ad28
Equity symbols loaded: 397
Nifty 50 rows loaded: 1913 (2019-01-02 to 2026-10-05)
Screen 2025-08-15: 388 stocks | Growth coverage: 0%
  Nifty benchmark: 24876.95 → 25879.15 = +4.03%
Screen 2025-11-15: 396 stocks | Growth coverage: 0%
  Nifty benchmark: 26013.45 → 25471.10 = -2.08%
Screen 2026-02-15: 397 stocks | Growth coverage: 2%
  Nifty benchmark: 25682.75 → 23649.95 = -7.92%
Screen 2026-05-15: 397 stocks | Growth coverage: 86%
  Nifty benchmark: 23643.50 → 24395.85 = +3.18%
Screen 2026-08-15: 397 stocks | Growth coverage: 94%
  ⚠️ Nifty benchmark MISSING for 2026-08-15 (entry=24287.650390625, exit=nan)

Backtest picks: 257 | Live picks: 75

--- picks per style per screen ---
screen_date  2025-08-15  2025-11-15  2026-02-15  2026-05-15  2026-08-15
style                                                                  
BUFFETT            15.0        15.0        15.0        15.0        15.0
GRAHAM             15.0        15.0        15.0     

SynapseWidget(Synapse.DataFrame, 0bbf1f45-f928-4e50-bd78-6add51110af5)

✅ LIVE forward portfolio saved: 75 picks (track until 2026-11-13)
